# RIT API — Agentic Workflow Test Suite

Tests every chat-capable model on the RIT API endpoint against a **multi-step tool-calling task** using a simple agent loop.

| Section | Purpose |
|---------|---------|
| §1 Config | API endpoint, model list |
| §2 Tools | 4 callable tools + schemas |
| §3 Agent loop | Core `run_agent()` function |
| §4 Single test | Quick sanity check for one model |
| §5 Batch test | Sweep all models, collect results |
| §6 Results | Summary table + save CSV |

> **Before running:** set `BASE_URL` in §1 to your RIT API endpoint.


## §1  Configuration

In [ ]:
import json, math, time, traceback
from datetime import datetime
from typing import Optional

from openai import OpenAI
import pandas as pd
from IPython.display import display, HTML
from dotenv import load_dotenv
import os

load_dotenv()
# ── API connection ─────────────────────────────────────────────────────────
BASE_URL  = "https://api.genai.gccis.rit.edu/v1"  
API_KEY   = os.getenv("RIT_API_KEY") 
TIMEOUT   = 120  

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=TIMEOUT)

# ── Model lists ────────────────────────────────────────────────────────────
ALL_MODELS = [
    "gemma4:26b",
    "gemma4:latest",
    "qwen3.6:35b-a3b-nvfp4",
    "cogito:70b",
    "qwen3.5:35b-a3b-coding-nvfp4",
    "qwen3.5:latest",
    "qwen3:32b",
    "llama3.1:70b",
    "smollm2:1.7b",
    "qwen3-coder:30b",
    "deepseek-r1:8b",
    "devstral:latest",
    "gpt-oss:120b",
    "qwen3:latest",
    "qwen3:1.7b",
    "qwen3:8b",
    "llava:latest",
    "qwen3-coder-next:latest",
    "qwen3.5:27b",
    "gemma3:27b",
]

# Skip embedding / OCR-only models – they don't do chat
SKIP_MODELS = {"embeddinggemma:latest", "nomic-embed-text:latest", "glm-ocr:latest"}
CHAT_MODELS = [m for m in ALL_MODELS if m not in SKIP_MODELS]

print(f"Total models:        {len(ALL_MODELS)}")
print(f"Skipped (non-chat):  {len(SKIP_MODELS)}")
print(f"Chat models to test: {len(CHAT_MODELS)}")


## §2  Tool Definitions

Four simple tools that cover common agentic patterns:
- **calculator** – math eval (tests numeric reasoning + tool use)
- **get_current_datetime** – no-arg tool (tests trivial tool dispatch)
- **count_words** – string tool (tests arg passing)
- **lookup_fact** – stub retrieval (tests open-ended lookup)


In [ ]:
# ── Python implementations ─────────────────────────────────────────────────

def calculator(expression: str) -> str:
    """Safely evaluate a math expression using Python's math module."""
    try:
        allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(round(float(result), 6))
    except Exception as e:
        return f"Error: {e}"


def get_current_datetime() -> str:
    """Return the current date and time as a string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def count_words(text: str) -> str:
    """Return the number of whitespace-separated words in text."""
    return str(len(text.split()))


def lookup_fact(topic: str) -> str:
    """Stub knowledge lookup – returns canned facts for a few topics."""
    db = {
        "python":    "Python was created by Guido van Rossum and first released in 1991.",
        "rit":       "Rochester Institute of Technology (RIT) was founded in 1829 and is "
                     "located in Henrietta, New York.",
        "pi":        "Pi (π) is approximately 3.14159265358979.",
        "einstein":  "Albert Einstein developed the theory of general relativity in 1915.",
        "ollama":    "Ollama is an open-source tool for running large language models locally.",
    }
    key = topic.lower().strip()
    for k, v in db.items():
        if k in key or key in k:
            return v
    return f"No fact found for '{topic}' in the stub knowledge base."


TOOL_REGISTRY = {
    "calculator":           calculator,
    "get_current_datetime": get_current_datetime,
    "count_words":          count_words,
    "lookup_fact":          lookup_fact,
}

# ── OpenAI-compatible JSON schemas ─────────────────────────────────────────

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a mathematical expression. "
                "Supports Python math functions: sqrt, sin, cos, log, pow, abs, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. 'sqrt(144) + 4'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": "Return the current date and time.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "count_words",
            "description": "Count the number of words in a text string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The text to count words in.",
                    }
                },
                "required": ["text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_fact",
            "description": "Look up a fact about a topic from a knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "Topic name, e.g. 'RIT', 'Python', 'Einstein'.",
                    }
                },
                "required": ["topic"],
            },
        },
    },
]

print(f"✅ {len(TOOLS)} tools defined: {[t['function']['name'] for t in TOOLS]}")
print("\nTool registry keys:", list(TOOL_REGISTRY.keys()))


## §3  Agent Loop

`run_agent()` implements a standard ReAct-style tool-calling loop:

```
user message → LLM → tool calls? → execute → append results → LLM → … → final answer
```

Returns a result dict regardless of success/failure so the batch runner never crashes.


In [ ]:
import re

def _strip_think_tags(text: str) -> str:
    """Strip <think>…</think> reasoning blocks (deepseek-r1, cogito, etc.)."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def _dispatch_tool(name: str, args: dict) -> str:
    """Execute a registered tool and return its string result."""
    if name not in TOOL_REGISTRY:
        return f"[ERROR] Unknown tool: '{name}'"
    try:
        fn = TOOL_REGISTRY[name]
        result = fn(**args)
        return str(result)
    except TypeError as e:
        return f"[ERROR] Bad arguments for {name}: {e}"
    except Exception as e:
        return f"[ERROR] {name} raised: {e}"


def run_agent(
    model: str,
    user_message: str,
    system_prompt: str = (
        "You are a helpful assistant. "
        "Use the provided tools whenever they would help answer the user's question. "
        "Always call the relevant tools before giving your final answer."
    ),
    tools: list = TOOLS,
    max_iterations: int = 10,
    temperature: float = 0.0,
    verbose: bool = True,
) -> dict:
    """
    Run one full agentic loop for a given model.

    Returns
    -------
    dict with keys:
        final_answer     : str  – model's last text response (think tags stripped)
        tool_calls_made  : list – [{name, args, result}, ...]
        iterations       : int
        total_time       : float (seconds)
        error            : str or None
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]
    tool_calls_made = []
    iterations = 0
    t0 = time.time()

    try:
        while iterations < max_iterations:
            iterations += 1
            if verbose:
                print(f"  [{iterations:02d}] → {model}")

            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools,
                tool_choice="auto",
                temperature=temperature,
            )
            choice = response.choices[0]
            msg    = choice.message

            # ── Build assistant message for history (explicit dict is safest) ──
            asst_msg: dict = {"role": "assistant", "content": msg.content or ""}
            if msg.tool_calls:
                asst_msg["tool_calls"] = [
                    {
                        "id":   tc.id,
                        "type": "function",
                        "function": {
                            "name":      tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in msg.tool_calls
                ]
            messages.append(asst_msg)

            # ── No tool calls → we have the final answer ────────────────────
            if not msg.tool_calls:
                final = _strip_think_tags(msg.content or "")
                if verbose:
                    print(f"  ✅ Done in {iterations} iteration(s), "
                          f"{len(tool_calls_made)} tool call(s)")
                return {
                    "final_answer":    final,
                    "tool_calls_made": tool_calls_made,
                    "iterations":      iterations,
                    "total_time":      round(time.time() - t0, 2),
                    "error":           None,
                }

            # ── Execute every tool call in this turn ────────────────────────
            for tc in msg.tool_calls:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}

                if verbose:
                    print(f"     🔧 {name}({args})")

                result = _dispatch_tool(name, args)

                if verbose:
                    print(f"        ↳ {result!r}")

                tool_calls_made.append({"name": name, "args": args, "result": result})

                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "name":         name,
                    "content":      result,
                })

        # Exhausted iterations without a final answer
        return {
            "final_answer":    "(max iterations reached without final answer)",
            "tool_calls_made": tool_calls_made,
            "iterations":      iterations,
            "total_time":      round(time.time() - t0, 2),
            "error":           "max_iterations_exceeded",
        }

    except Exception as exc:
        if verbose:
            traceback.print_exc()
        return {
            "final_answer":    None,
            "tool_calls_made": tool_calls_made,
            "iterations":      iterations,
            "total_time":      round(time.time() - t0, 2),
            "error":           str(exc),
        }


print("✅ run_agent() ready")


## §4  Test Tasks

Define one or more tasks. The default task requires **3 tool calls** in sequence:
1. `calculator` — `sqrt(144) + count of words in a phrase`
2. `get_current_datetime` — no args
3. `lookup_fact` — topic lookup

The expected numeric answer is **16.0** (√144 = 12, "the quick brown fox" = 4 words → 12+4).


In [ ]:
TASKS = {
    # Three steps that must each use a separate tool call.
    # Expected numeric answer: sqrt(144)=12, count_words("the quick brown fox")=4 → sum=16
    "multi_tool": (
        "Please complete all three steps below using the available tools.\n"
        "Step 1a: Call count_words with the text 'the quick brown fox' to get the word count.\n"
        "Step 1b: Call calculator with 'sqrt(144) + <word count from step 1a>' to get the sum.\n"
        "Step 2:  Call get_current_datetime to get today's date and time.\n"
        "Step 3:  Call lookup_fact with topic 'RIT'.\n"
        "Finally, report all results. The correct sum in step 1b should be 16."
    ),
    "math_only": (
        "What is sin(pi/6) + cos(pi/3)?  Use the calculator tool."
    ),
    "single_lookup": (
        "What do you know about Python (the programming language)? "
        "Use the lookup_fact tool."
    ),
}

# Default task used by the batch sweep
DEFAULT_TASK = TASKS["multi_tool"]
print("Default task:\n")
print(DEFAULT_TASK)


### Quick single-model test

In [ ]:
# ── Change this to test any single model quickly ──────────────────────────
QUICK_TEST_MODEL = "qwen3:8b"

print(f"Testing: {QUICK_TEST_MODEL}")
print("=" * 60)

result = run_agent(QUICK_TEST_MODEL, DEFAULT_TASK, verbose=True)

print("\n" + "=" * 60)
print("FINAL ANSWER:\n")
print(result["final_answer"])
print(f"\nTool calls : {result['tool_calls_made']}")
print(f"Iterations : {result['iterations']}")
print(f"Time       : {result['total_time']}s")
if result["error"]:
    print(f"Error      : {result['error']}")


## §5  Batch Sweep — All Models

Runs every chat model sequentially and collects a structured result row per model.
Set `PARALLEL = True` (below) to use a thread pool for faster sweeps (watch rate limits).


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _test_one(model: str, task: str, verbose: bool) -> dict:
    result = run_agent(model, task, verbose=verbose)
    return {"model": model, **result}


def run_all_models(
    task: str        = DEFAULT_TASK,
    models: list     = CHAT_MODELS,
    verbose: bool    = False,
    parallel: bool   = False,
    max_workers: int = 4,
) -> pd.DataFrame:
    """
    Sweep all models and return a results DataFrame.

    Columns
    -------
    model, status, error, tool_calls, iterations, time_s,
    answer_len, answer_preview
    """
    rows  = []
    total = len(models)

    if parallel:
        futures = {}
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for m in models:
                futures[ex.submit(_test_one, m, task, verbose)] = m
            for i, fut in enumerate(as_completed(futures), 1):
                m   = futures[fut]
                res = fut.result()
                ok  = res["error"] is None
                print(f"[{i:2d}/{total}] {'✅' if ok else '❌'}  {m}  "
                      f"({res['total_time']}s, {len(res['tool_calls_made'])} calls)")
                rows.append(_make_row(m, res))
    else:
        for i, m in enumerate(models, 1):
            print(f"[{i:2d}/{total}] {m} … ", end="", flush=True)
            res = run_agent(m, task, verbose=verbose)
            ok  = res["error"] is None
            print(f"{'✅' if ok else '❌'}  "
                  f"({res['total_time']}s, {len(res['tool_calls_made'])} tool calls)")
            rows.append(_make_row(m, res))

    return pd.DataFrame(rows)


def _make_row(model: str, res: dict) -> dict:
    ans = res["final_answer"] or ""
    return {
        "model":          model,
        "status":         "ok" if res["error"] is None else "error",
        "error":          (res["error"] or "")[:120],
        "tool_calls":     len(res["tool_calls_made"]),
        "iterations":     res["iterations"],
        "time_s":         res["total_time"],
        "answer_len":     len(ans),
        "answer_preview": ans[:150].replace("\n", " "),
    }


print("✅ Batch runner ready")
print(f"   Models queued: {len(CHAT_MODELS)}")
print(f"   Task: {DEFAULT_TASK[:60]}…")


In [ ]:
# ── Run the sweep ──────────────────────────────────────────────────────────
# Set PARALLEL=True to run concurrently (faster but may overwhelm the server)
PARALLEL = False

df = run_all_models(
    task        = DEFAULT_TASK,
    models      = CHAT_MODELS,
    verbose     = False,
    parallel    = PARALLEL,
    max_workers = 4,
)

print(f"\nDone. {(df.status=='ok').sum()}/{len(df)} models succeeded.")


## §6  Results

In [ ]:
# ── Styled summary table ───────────────────────────────────────────────────

def show_results(df: pd.DataFrame):
    ok  = df[df.status == "ok"]
    err = df[df.status == "error"]

    print(f"✅ {len(ok)} succeeded   ❌ {len(err)} failed\n")

    def row_style(row):
        color = "#d4edda" if row.status == "ok" else "#f8d7da"
        return [f"background-color: {color}"] * len(row)

    styled = (
        df[["model","status","tool_calls","iterations","time_s","answer_len","error"]]
        .style
        .apply(row_style, axis=1)
        .format({"time_s": "{:.1f}s"})
        .set_caption("RIT API — Agentic Workflow Results")
    )
    display(styled)

    if len(err):
        print("\n── Error details ──────────────────────────────────────")
        for _, r in err.iterrows():
            print(f"  {r.model:<40}  {r.error}")


show_results(df)


In [ ]:
# ── Quick stats ────────────────────────────────────────────────────────────
ok = df[df.status == "ok"]
if len(ok):
    print(f"Tool-calling support: {(ok.tool_calls > 0).sum()}/{len(ok)} models used tools")
    print(f"Avg time (ok):        {ok.time_s.mean():.1f}s  (min {ok.time_s.min():.1f}s, max {ok.time_s.max():.1f}s)")
    print(f"Avg tool calls:       {ok.tool_calls.mean():.1f}")
    print()
    print("Fastest models:")
    display(ok.nsmallest(5, "time_s")[["model","time_s","tool_calls","answer_len"]])


# ── Hallucination check (multi_tool task only) ──────────────────────────────
# Expected: answer contains "16" (sqrt(144)=12, 4 words → 12+4=16)
if "multi_tool" in str(DEFAULT_TASK):
    ok = df[df.status == "ok"].copy()
    ok["answer_correct"] = ok["answer_preview"].str.contains(r"\b16\b", regex=True)
    print("\nHallucination check — models that got the correct sum (16):")
    display(ok[["model","answer_correct","tool_calls","time_s"]])


In [ ]:
# ── Save results to CSV ────────────────────────────────────────────────────
ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"rit_agent_results_{ts}.csv"
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")


---
## Tips

- **Model doesn't support tool calling** → you'll see an `error` row; the model likely returns a plain text response instead of structured tool calls. You can retest it with `run_agent(model, task, tools=[], verbose=True)` to at least check connectivity.
- **Timeout errors** → large models (cogito:70b, gpt-oss:120b) may be slow; increase `TIMEOUT` in §1.
- **`max_iterations_exceeded`** → the model is looping tool calls; lower `max_iterations` or adjust the system prompt.
- **`<think>` blocks** → automatically stripped by `_strip_think_tags()` for deepseek-r1 / cogito style models.
- **Parallel mode** → set `PARALLEL = True` in §5 to speed up the sweep, but watch your server load.
